Thiết lập môi trường làm việc trong Google Colab để truy cập dữ liệu từ Google Drive, đồng thời import các thư viện cần thiết cho quá trình tiền xử lý dữ liệu

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import gc  # for memory cleanup

Mounted at /content/drive


Thay thế các giá trị rỗng ('') bằng NaN để biểu diễn “không có danh mục cha”

In [2]:
cat_path = '/content/drive/MyDrive/DW_data/category_tree.csv'

category_tree = pd.read_csv(cat_path, names=['categoryid', 'parentid'], header=0)
category_tree['parentid'] = category_tree['parentid'].replace({'': np.nan}).astype('float').astype('Int64')
print(category_tree.info())
category_tree.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1669 entries, 0 to 1668
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   categoryid  1669 non-null   int64
 1   parentid    1644 non-null   Int64
dtypes: Int64(1), int64(1)
memory usage: 27.8 KB
None


,categoryid,parentid
0,1016,213
1,809,169
2,570,9
3,1691,885
4,536,1691


Chuyển đổi timestamp sang dạng thời gian thực.

Một số dòng không có mã giao dịch (NaN) do không phải là sự kiện mua hàng (ví dụ: view, addtocart).

fillna(0): thay thế các giá trị NaN bằng 0 để biểu diễn “không có giao dịch”

In [3]:
events_path = '/content/drive/MyDrive/DW_data/events.csv'

events = pd.read_csv(
    events_path,
    names=['timestamp', 'visitorid', 'event', 'itemid', 'transactionid'],
    header=0,
    dtype={
        'visitorid': 'Int64',
        'itemid': 'Int64',
        'event': 'category',
        'transactionid': 'Int64'
    }
)

# Convert timestamp to datetime
events['timestamp'] = pd.to_datetime(events['timestamp'], unit='ms')

# Fill NaNs for transactionid
events['transactionid'] = events['transactionid'].fillna(0).astype('Int64')

print(events.info())
events.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 5 columns):
 #   Column         Dtype         
---  ------         -----         
 0   timestamp      datetime64[ns]
 1   visitorid      Int64         
 2   event          category      
 3   itemid         Int64         
 4   transactionid  Int64         
dtypes: Int64(3), category(1), datetime64[ns](1)
memory usage: 94.6 MB
None


,timestamp,visitorid,event,itemid,transactionid
0,2015-06-02 05:02:12.117,257597,view,355908,0
1,2015-06-02 05:50:14.164,992329,view,248676,0
2,2015-06-02 05:13:19.827,111016,view,318965,0
3,2015-06-02 05:12:35.914,483717,view,253185,0
4,2015-06-02 05:02:17.106,951259,view,367447,0


Đọc và hợp nhất hai phần dữ liệu item_properties

Chuyển đổi định dạng thời gian

In [4]:
part1_path = '/content/drive/MyDrive/DW_data/item_properties_part1.csv'
part2_path = '/content/drive/MyDrive/DW_data/item_properties_part2.csv'

def load_properties(file_path):
    return pd.read_csv(
        file_path,
        names=['timestamp', 'itemid', 'property', 'value'],
        header=0,
        dtype={'itemid': 'Int64', 'property': 'string', 'value': 'string'}
    )

item_prop1 = load_properties(part1_path)
item_prop2 = load_properties(part2_path)

# Combine parts
item_properties = pd.concat([item_prop1, item_prop2], ignore_index=True)
del item_prop1, item_prop2
gc.collect()

# Convert timestamp to datetime
item_properties['timestamp'] = pd.to_datetime(item_properties['timestamp'], unit='ms')

print(item_properties.info())
item_properties.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20275902 entries, 0 to 20275901
Data columns (total 4 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   itemid     Int64         
 2   property   string        
 3   value      string        
dtypes: Int64(1), datetime64[ns](1), string(2)
memory usage: 638.1 MB
None


,timestamp,itemid,property,value
0,2015-06-28 03:00:00,460429,categoryid,1338
1,2015-09-06 03:00:00,206783,888,1116713 960601 n277.200
2,2015-08-09 03:00:00,395014,400,n552.000 639502 n720.000 424566
3,2015-05-10 03:00:00,59481,790,n15360.000
4,2015-05-17 03:00:00,156781,917,828513


Từ bảng dữ liệu thuộc tính sản phẩm (item_properties), trích xuất hai thông tin quan trọng:

categoryid – danh mục sản phẩm.

available – trạng thái còn hàng hay không.
Sau đó hợp nhất chúng thành một bảng duy nhất item_info

In [5]:
# Extract categoryid info
item_categories = item_properties[item_properties['property'] == 'categoryid'].copy()
item_categories['value'] = pd.to_numeric(item_categories['value'], errors='coerce').astype('Int64')

# Extract availability info
item_avail = item_properties[item_properties['property'] == 'available'].copy()
item_avail['value'] = item_avail['value'].astype('Int64')

# Merge these two
item_info = (
    item_categories[['itemid', 'value']]
    .rename(columns={'value': 'categoryid'})
    .merge(
        item_avail[['itemid', 'value']],
        on='itemid',
        how='left'
    )
    .rename(columns={'value': 'available'})
)

In [6]:
item_info = item_info.astype(str).drop_duplicates()
item_info = item_info.reset_index(drop=True)

item_info['available'] = item_info['available'].astype('Int64')
item_info['categoryid'] = item_info['categoryid'].astype('Int64')
item_info['itemid'] = item_info['itemid'].astype('Int64')


In [7]:
print(item_info.info())
item_info.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 513503 entries, 0 to 513502
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   itemid      513503 non-null  Int64
 1   categoryid  513503 non-null  Int64
 2   available   513503 non-null  Int64
dtypes: Int64(3)
memory usage: 13.2 MB
None


,itemid,categoryid,available
0,460429,1338,0
1,281245,1277,0
2,35575,1059,0
3,8313,1147,0
4,55102,47,0


Gộp các bảng dữ liệu

In [8]:
# Merge events with item info
events_merged = events.merge(item_info, on='itemid', how='left')

# Merge category tree if needed
events_final = events_merged.merge(category_tree, on='categoryid', how='left')

print(events_final.info())
events_final.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4259061 entries, 0 to 4259060
Data columns (total 8 columns):
 #   Column         Dtype         
---  ------         -----         
 0   timestamp      datetime64[ns]
 1   visitorid      Int64         
 2   event          category      
 3   itemid         Int64         
 4   transactionid  Int64         
 5   categoryid     Int64         
 6   available      Int64         
 7   parentid       Int64         
dtypes: Int64(6), category(1), datetime64[ns](1)
memory usage: 255.9 MB
None


,timestamp,visitorid,event,itemid,transactionid,categoryid,available,parentid
0,2015-06-02 05:02:12.117,257597,view,355908,0,1173,1,805
1,2015-06-02 05:02:12.117,257597,view,355908,0,1173,0,805
2,2015-06-02 05:50:14.164,992329,view,248676,0,1231,1,901
3,2015-06-02 05:13:19.827,111016,view,318965,0,<NA>,<NA>,<NA>
4,2015-06-02 05:12:35.914,483717,view,253185,0,914,0,226


In [ ]:
# events_final = events_final.astype(str).drop_duplicates()
# events_final = events_final.reset_index(drop=True)

In [ ]:
events_final.to_csv('/content/drive/MyDrive/DW_data/events_cleaned.csv', index=False)
item_info.to_csv('/content/drive/MyDrive/DW_data/item_info_cleaned.csv', index=False)
category_tree.to_csv('/content/drive/MyDrive/DW_data/category_tree_cleaned.csv', index=False)